In [ ]:
# Dependencies and imports - RUN THIS CELL FIRST
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets
from datetime import datetime
import threading
from DS1104_interface_V2 import ThermalControlApparatus, CollectMeasurementData, AnimateScope

# Start & connect thermal apparatus - this can take up to 30 seconds
plant = ThermalControlApparatus() # Load thermal control apparatus interface
plant.start() # Start the apparatus (connects and starts safty thread)   

### Press "Start Measuring" in D-Space Before continuing!

![alt text](start_measuring_dspace.png "Start Measurement Procedure")

## Define plant model

In [ ]:
import control as ct

num = [21]
den = [370.0, 1.0]
dead_time = 5.0
def create_transferFunction(numerator, denominator):
    # dead_time_TF = ct.tf([1/2*dead_time, 1], [-1/2*dead_time, 1])
    plant = ct.tf(numerator, denominator)
    num_DT, den_DT = ct.pade(dead_time, 4)
    dead_time_TF = ct.tf(num_DT, den_DT)

    return ct.series(plant, dead_time_TF)

plant_model = create_transferFunction(num, den)

## Scope Animation (tracking signals)

In [ ]:
ani = None # Global variable to hold the animation object
plt.close("all") # Close all previous figures to avoid overlap

# Plot settings:
window_length = 30 # The length of the window size that is shown in the plot, in seconds
dt = 0.5/10 # Interval that controls when a new frame is generated, don't go lower than dt <  0.01 to perserve CPU

def run_T_scope():
    global ani
    if ani is not None:
        ani.event_source.stop()  # Stop any other animation that is running

    plt.close("all")  # Close all previous figures to avoid overlap
    # Create two subplots: one for temperatures, one for inputs
    fig, (ax_temp, ax_input) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Temperature lines
    line1, = ax_temp.plot(np.linspace(dt, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_1')
    line2, = ax_temp.plot(np.linspace(dt, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_2')
    line_model, = ax_temp.plot(np.linspace(dt, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_model', linestyle='--')
    
    ax_temp.legend()
    ax_temp.set_ylim(0, 100)
    ax_temp.set_title("Live Temperature Readings")
    ax_temp.set_ylabel("Temperature (°C)")
    ax_temp.grid(True)

    # Input lines
    line3, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Heater')
    line4, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Fan')
    line5, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Vane')
    
    ax_input.legend()
    ax_input.set_ylim(0, 12)
    ax_input.set_title("Supplied Inputs")
    ax_input.set_xlabel("Time (s)")
    ax_input.set_ylabel("Input Value")
    ax_input.grid(True)
    # Use the new AnimateScope signature
    state = AnimateScope([line1, line2, line_model], [line3, line4, line5], ax_temp, ax_input, window_length=window_length, dt=dt, plant=plant, plant_model=plant_model)
    ani = animation.FuncAnimation(fig,
                                 state.animate,
                                #  init_func=state.init_plot,
                                 interval=dt*1000, #convert the framerate dt to how often the plot is called
                                 blit=True,
                                 save_count=0)

    plt.tight_layout()
    # plt.show()

run_T_scope()


## Sliders to control the inputs

In [ ]:
HeatSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Heater Power')
FanSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Fan Power')
VaneSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Vane Rotation')

widgets.interact(plant.write_heater, heater_input=HeatSlider);
widgets.interact(plant.write_fan, fan_input=FanSlider);
widgets.interact(plant.write_vane, vane_input=VaneSlider);

## Start or stop measurements & save to CSV

In [ ]:
# Call the data_collector object, which enables the logging of data to a CSV file, 
# The CSV is stored in "Z:\Exp_Data_Out", if this path is not reachable, it is instead stored in "C:\Thermal\Project_Thermal\ExperimentData"
data_collector = CollectMeasurementData(plant, fs=1.0, Exp_length_minutes=1.0)

### Controller Building: 

## Control loop + Measurment

In [ ]:
# Run control loop in a separate thread to keep widgets responsive
T_ref = 40.0  # Desired temperature reference
control_duration = 90  # Control loop duration in seconds
dt = 0.1  # Control loop time step in seconds
data_collector_ctrl = CollectMeasurementData(plant, fs=1/dt, Exp_length_minutes=control_duration/60, display_button=False)

def controller_PID(T_current, T_ref = T_ref, dt = dt):
        # Function for a proportional control law with integral term
        Kp_heater = 10  # Proportional gain for heater (tune as needed)
        KI_heater = 0  # Integral gain (tune as needed)
        Kp_fan =    10  # Proportional gain for fan

        # heater section
        integral_error = 0
        error = T_ref - T_current
        integral_error += error * dt  # Update integral error

        heater_power = Kp_heater * error + KI_heater * integral_error  # PI control law
        fan_power = Kp_fan * error
        vane_rotation = 0
        
        # Saturate heater power to valid range [0, 10]
        unsaturated_output_powers = [heater_power, fan_power, vane_rotation]
        output_powers = [max(0, min(10, u_in)) for u_in in unsaturated_output_powers]
        return output_powers[0], output_powers[1], output_powers[2]


## DO NOT TOUCH THIS PART OF THIS CELL
thermal_control_loop = ThermalControlApparatus().control_loop
controller_thread = threading.Thread(target=thermal_control_loop, args=(plant, controller_PID, control_duration, dt))
data_collector_ctrl.start()
controller_thread.start()


In [76]:
## After running the control loop, you can stop the plant
plant.stop()

In [ ]:
# # Run control loop in a separate thread to keep widgets responsive
# T_ref = 40.0  # Desired temperature reference
# control_duration = 90  # Control loop duration in seconds
# dt = 0.1  # Control loop time step in seconds
# def controller_PID(T_current, T_ref = T_ref, dt = dt):
#         # Function for a proportional control law with integral term
#         Kp_heater = 100  # Proportional gain for heater (tune as needed)
#         KI_heater = 0  # Integral gain (tune as needed)
#         Kp_fan =    0  # Proportional gain for fan

#         # heater section
#         integral_error = 0
#         error = T_ref - T_current
#         integral_error += error * dt  # Update integral error

#         heater_power = Kp_heater * error + KI_heater * integral_error  # PI control law
#         fan_power = Kp_fan * error
#         vane_rotation = 0
#         # Saturate heater power to valid range [0, 10]
#         unsaturated_output_powers = [heater_power, fan_power, vane_rotation]
#         output_powers = [max(0, min(10, u_in)) for u_in in unsaturated_output_powers]
#         return output_powers[0], output_powers[1], output_powers[2]


# ## DO NOT TOUCH THIS PART OF THIS CELL
# thermal_control_loop = ThermalControlApparatus().control_loop
# controller_thread = threading.Thread(target=thermal_control_loop, args=(plant, controller_PID, control_duration, dt))
# controller_thread.start()

In [ ]:
# def plant_model_test(t, u):
#     # DO NOT MODIFY THIS SECTION
#     u_heater  = u[0]  # Heater input, in the range [0, 10] V
#     u_fan     = u[1]  # Fan input , in the range [0, 10] V
#     u_vane    = u[2]  # Vane Rotation, can be fully open (0 V) or fully closed (10 V)

#     # Insert your plant model here. For example, a simple first-order system could be modeled as:
#     T_env=23
#     A=350.0
#     B=32.75/1.5
#     t0=0.0
#     T = B*u_heater*(1-np.exp(-(1/A*(t - 5 - t0)))) + T_env
#     return T